# DINO Keypoint Matching

Register named keypoints on a reference image using a `Gallery` + `KeypointHead`,
then locate those same keypoints in a set of query images (rotations, flips, etc.)
via nearest-patch cosine similarity.

**Workflow**
1. Build a `Gallery` from the reference image.
2. Register pixel coordinates + labels with `KeypointHead.register()`.
3. Call `KeypointHead.find()` on each query — one encoder pass per query.
4. Visualise reference (marked) vs. queries (predicted) side by side.

In [ ]:
# Logging must be configured before torch is imported.
import logging
import os

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(name)s: %(message)s",
    force=True,
)
log = logging.getLogger("keypoint_matching")

import tempfile
from pathlib import Path

import cv2
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from dotenv import load_dotenv

from dinoisawesome import DinoEncoder, Gallery, GalleryConfig, KeypointHead

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
load_dotenv()

# Reference image — set to a local file path string.
REF_IMAGE_PATH: str = "experimenting/testdata/corner_good.png"
TAR_IMAGE_PATH: str = ["experimenting/testdata/corner_bad.png", "experimenting/testdata/corner_bad2.png"]

# Query transforms applied to the reference image to create synthetic queries.
# Each entry is (display_name, transform_fn).
QUERIES: list[tuple[str, object]] = [
    ("identity",      lambda img: img),
    ("rotate +15°",   lambda img: img.rotate(15,  expand=False, fillcolor=(128, 128, 128))),
    ("rotate -30°",   lambda img: img.rotate(-30, expand=False, fillcolor=(128, 128, 128))),
    ("flip H",        lambda img: img.transpose(Image.FLIP_LEFT_RIGHT)),
    ("flip V",        lambda img: img.transpose(Image.FLIP_TOP_BOTTOM)),
    ("rotate +90°",   lambda img: img.rotate(90,  expand=True)),
]

# Model settings
DINO_VERSION = "v3"   # "v2" or "v3"
DINO_SIZE    = "large" # "small" | "base" | "large" | "giant"
IMG_SIZE     = 1024    # square input side (divisible by patch_size: 14 for v2, 16 for v3)
DEBIAS       = True   # remove positional subspace before matching

# Local weights directory — set DINO_WEIGHTS_DIR in the environment to load
# pre-downloaded .pth files instead of fetching from torch hub.
DINO_WEIGHTS_DIR: str | None = os.environ.get("DINO_WEIGHTS_DIR")

# Marker style for visualisation
MARKER_RADIUS  = 6
MARKER_COLORS  = [  # one colour per keypoint label
    (255,  60,  60),   # red
    ( 60, 180,  60),   # green
    ( 60, 100, 255),   # blue
    (255, 180,   0),   # orange
    (180,   0, 255),   # purple
    (  0, 210, 210),   # cyan
]

In [ ]:
# ── Load encoder ───────────────────────────────────────────────────────────────
encoder = DinoEncoder(
    version=DINO_VERSION,
    size=DINO_SIZE,
    img_size=IMG_SIZE,
    weights_dir=DINO_WEIGHTS_DIR,
)
log.info(
    "DINOv%s-%s | patch_size=%d | grid=%dx%d",
    DINO_VERSION[1], DINO_SIZE,
    encoder.patch_size, encoder.grid_h, encoder.grid_w,
)

In [ ]:
# ── Load reference image ───────────────────────────────────────────────────────
ref_img = Image.open(REF_IMAGE_PATH).convert("RGB")
orig_w, orig_h = ref_img.size
log.info("Reference image: %s  (%dx%d px)", Path(REF_IMAGE_PATH).name, orig_w, orig_h)

# Keypoints to register: list of (x, y) pixel coords + label.
# Coords are in the **original** image pixel space (before any resize).
KEYPOINTS: list[dict] = [
    {"x": 230, "y": 65, "label": "point2"},
    {"x": 150, "y": 210, "label": "point1"},
    {"x": 280, "y": 325, "label": "point3"},
    {"x": 140, "y": 380, "label": "point4"},
]

fig, ax = plt.subplots(figsize=(6, 6))
canvas = np.array(ref_img)
for i, kp in enumerate(KEYPOINTS):
    x, y = kp["x"], kp["y"]
    color = MARKER_COLORS[i % len(MARKER_COLORS)]
    cv2.circle(canvas, (x, y), MARKER_RADIUS, color, thickness=-1)
ax.imshow(canvas)
ax.set_title(f"Reference image  ({orig_w}×{orig_h} px)")

plt.tight_layout()
plt.show()

In [ ]:
# ── Build gallery from reference image ────────────────────────────────────────
_gallery_dir = Path(tempfile.mkdtemp(prefix="dino_kp_gallery_"))
log.info("Building gallery in %s …", _gallery_dir)

gallery = Gallery.build(
    encoder=encoder,
    images=[ref_img],
    image_ids=["ref"],
    out_dir=_gallery_dir,
    split="train",
)
log.info(
    "Gallery built: %d patches, blocks=%s",
    len(gallery.patches),
    gallery.config.block_indices,
)

In [ ]:
# ── Register keypoints ────────────────────────────────────────────────────────
head = KeypointHead(gallery, encoder)

points = [[kp["x"], kp["y"]] for kp in KEYPOINTS]
labels = [kp["label"]          for kp in KEYPOINTS]

head.register(
    image_id="ref",
    points=points,
    labels=labels,
    orig_size=(orig_w, orig_h),
)
head.save()
log.info("Registered keypoints: %s", head.registered_labels)

In [ ]:
# ── Visualise reference image with registered keypoints ───────────────────────
label_color = {lbl: MARKER_COLORS[i % len(MARKER_COLORS)] for i, lbl in enumerate(labels)}

def draw_points(
    img: Image.Image,
    pts: list[tuple[int, int]],
    pt_labels: list[str],
    show_labels: bool = True,
) -> np.ndarray:
    """Return a numpy canvas with circles drawn for each (x, y) point."""
    canvas = np.array(img.convert("RGB"))
    for (x, y), lbl in zip(pts, pt_labels):
        color = label_color[lbl]
        cv2.circle(canvas, (int(x), int(y)), MARKER_RADIUS, color, thickness=-1)
        cv2.circle(canvas, (int(x), int(y)), MARKER_RADIUS, (255, 255, 255), thickness=2)
        if show_labels:
            cv2.putText(
                canvas, lbl,
                (int(x) + MARKER_RADIUS + 4, int(y) + 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA,
            )
    return canvas


ref_canvas = draw_points(ref_img, [(kp["x"], kp["y"]) for kp in KEYPOINTS], labels)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(ref_canvas)
legend_handles = [
    mpatches.Patch(color=[c / 255 for c in label_color[lbl]], label=lbl)
    for lbl in labels
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=9, framealpha=0.85)
ax.set_title("Reference image — registered keypoints")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ── Run KeypointHead.find() on all query images ───────────────────────────────
query_results: list[dict] = []
for tar_path in TAR_IMAGE_PATH:
    tar_img = Image.open(tar_path).convert("RGB")
    log.info("Target image: %s  (%dx%d px)", Path(tar_path).name, *tar_img.size)
    for name, transform in QUERIES:
        q_img = transform(tar_img)
        matches = head.find(q_img, labels=labels, debias=DEBIAS)
        query_results.append({"name": name, "image": q_img, "matches": matches})
        for m in matches:
            log.info("  [%s] %s → (%d, %d)  sim=%.3f",
                    name, m["label"], m["point"][0], m["point"][1], m["similarity"])

log.info("Done — %d queries × %d keypoints", len(query_results), len(labels))

In [ ]:
# ── Grid visualisation: reference  +  one column per query ───────────────────
NCOLS = len(query_results)//len(TAR_IMAGE_PATH)
NROWS = len(TAR_IMAGE_PATH) + 1
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(NCOLS * 5, NROWS * 5))

# Reference column
axes[0, 0].imshow(ref_canvas)
axes[0, 0].set_title("Reference\n(registered)", fontsize=10)
axes[0, 0].axis("off")

for row, tar_path in enumerate(TAR_IMAGE_PATH, start=1):
    
    for col, qr in enumerate(query_results[row-1::len(TAR_IMAGE_PATH)], start=0):
        q_pts    = [m["point"] for m in qr["matches"]]
        q_labels = [m["label"] for m in qr["matches"]]
        canvas   = draw_points(qr["image"], q_pts, q_labels, show_labels=False)
        axes[row, col].imshow(canvas)
        axes[row, col].set_title(qr["name"], fontsize=10)
        axes[row, col].axis("off")

legend_handles = [
    mpatches.Patch(color=[c / 255 for c in label_color[lbl]], label=lbl)
    for lbl in labels
]
fig.legend(handles=legend_handles, loc="lower center", ncol=len(labels),
           fontsize=9, framealpha=0.85, bbox_to_anchor=(0.5, -0.04))
plt.suptitle(
    f"KeypointHead  |  DINOv{DINO_VERSION[1]}-{DINO_SIZE}  |  "
    f"img_size={IMG_SIZE}  |  debias={DEBIAS}",
    fontsize=12, y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-keypoint similarity scores across queries ─────────────────────────────
# Bar chart: one group per label, one bar per query.
import numpy as np

n_queries  = len(query_results)
n_labels   = len(labels)
bar_width  = 0.8 / n_queries
x          = np.arange(n_labels)

fig, ax = plt.subplots(figsize=(max(8, n_labels * 1.5), 4))

for q_idx, qr in enumerate(query_results):
    sims = [
        next((m["similarity"] for m in qr["matches"] if m["label"] == lbl), 0.0)
        for lbl in labels
    ]
    offset = (q_idx - n_queries / 2 + 0.5) * bar_width
    ax.bar(x + offset, sims, bar_width, label=qr["name"], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("Cosine similarity")
ax.set_ylim(0, 1.05)
ax.set_title("Per-keypoint cosine similarity across query transforms")
ax.legend(loc="lower right", fontsize=8, framealpha=0.85)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Similarity heatmaps for each keypoint on a chosen query ──────────────────
# Shows the full cosine-similarity map between the stored keypoint embedding
# and every patch in the query, so you can see where the model is "looking".

QUERY_IDX = 0   # which query to inspect (index into QUERIES list)

import torch
import torch.nn.functional as F

qr        = query_results[QUERY_IDX]
q_img     = qr["image"]
q_w, q_h  = q_img.size

# Re-extract query patches (single forward pass, all blocks).
out    = encoder([q_img], layers=[head.block_idx], debias=DEBIAS)
patches = out.patches[0, 0]            # (H, W, D)
H, W, D = patches.shape
flat    = F.normalize(patches.reshape(H * W, D), p=2, dim=1)  # (N, D)

display_q = np.array(q_img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC))

n_kp   = len(labels)
NCOLS2 = n_kp
fig, axes = plt.subplots(2, NCOLS2, figsize=(NCOLS2 * 4, 8))
if NCOLS2 == 1:
    axes = axes.reshape(2, 1)

for i, lbl in enumerate(labels):
    ref_emb = head._get_label_emb(lbl)  # (D,)
    if ref_emb is None:
        axes[0, i].axis("off")
        axes[1, i].axis("off")
        continue

    sims   = (flat @ ref_emb.to(flat.device)).reshape(H, W).cpu().numpy()  # (H, W)
    # Upsample heatmap to display resolution for overlay
    heat_pil   = Image.fromarray(((sims - sims.min()) / (sims.max() - sims.min() + 1e-8) * 255)
                                  .astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    heat_np    = np.array(heat_pil)
    heat_color = plt.get_cmap("jet")(heat_np / 255.0)[..., :3]  # (H, W, 3)
    blended    = (0.45 * display_q / 255.0 + 0.55 * heat_color)
    blended    = np.clip(blended, 0, 1)

    # Top row: raw heatmap
    im = axes[0, i].imshow(sims, cmap="jet", vmin=sims.min(), vmax=1.0, aspect="auto")
    axes[0, i].set_title(f"{lbl}\n(patch sim map)", fontsize=9)
    axes[0, i].axis("off")
    plt.colorbar(im, ax=axes[0, i], shrink=0.75, pad=0.02)

    # Bottom row: blended overlay with predicted point
    axes[1, i].imshow(blended)
    m = next(m for m in qr["matches"] if m["label"] == lbl)
    px_scaled = int(m["point"][0] * IMG_SIZE / q_w)
    py_scaled = int(m["point"][1] * IMG_SIZE / q_h)
    color_f   = [c / 255 for c in label_color[lbl]]
    axes[1, i].add_patch(plt.Circle((px_scaled, py_scaled), MARKER_RADIUS,
                                     color=color_f, zorder=5))
    axes[1, i].set_title(f"sim={m['similarity']:.3f}", fontsize=9)
    axes[1, i].axis("off")

plt.suptitle(
    f"Similarity heatmaps — query: \"{qr['name']}\"  |  "
    f"block={head.block_idx}  debias={DEBIAS}",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()

## Homography & Perspective Transform

Use the keypoint correspondences (reference → query) to estimate a homography with RANSAC,
then warp the reference image into the query's coordinate frame.

In [ ]:
# ── Estimate homography from keypoint correspondences ─────────────────────────
# query_results is ordered: [tar0×transform0, tar0×transform1, ..., tar1×transform0, ...]
# Pick a single query by flat index.
#HOMO_QUERY_IDX = 6   # index into query_results
HOMO_QUERY_IDX = 0   # index into query_results

qr    = query_results[HOMO_QUERY_IDX]
q_img = qr["image"]
q_w, q_h = q_img.size

# Source points: original registered pixel coords on the reference image.
# Destination points: predicted pixel coords found by head.find() on the query.
match_map  = {m["label"]: m["point"] for m in qr["matches"]}
valid_kps  = [kp for kp in KEYPOINTS if kp["label"] in match_map]
src_pts    = np.array([[kp["x"], kp["y"]] for kp in valid_kps], dtype=np.float32)
dst_pts    = np.array([match_map[kp["label"]] for kp in valid_kps], dtype=np.float32)
valid_labels = [kp["label"] for kp in valid_kps]

if len(src_pts) < 4:
    log.warning("Need at least 4 correspondences for homography; got %d.", len(src_pts))
    H_mat, inlier_mask = None, None
else:
    H_mat, inlier_mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, ransacReprojThreshold=8.0)
    n_inliers = int(inlier_mask.sum()) if inlier_mask is not None else 0
    log.info(
        "Query: %r | correspondences=%d | RANSAC inliers=%d",
        qr["name"], len(src_pts), n_inliers,
    )
    log.info("Homography matrix:\n%s", np.array2string(H_mat, precision=4, suppress_small=True))

In [ ]:
# ── Warp reference → query frame and visualise ────────────────────────────────
if H_mat is None:
    log.warning("No homography — skipping warp visualisation.")
else:
    ref_np = np.array(ref_img)
    q_np   = np.array(q_img)
    h_out, w_out = q_np.shape[:2]

    # Warp the reference image into the query coordinate frame.
    warped = cv2.warpPerspective(ref_np, H_mat, (w_out, h_out))

    # Blend warped reference over the query (only where the warp produced content).
    WARP_ALPHA  = 0.45
    blend_mask  = (warped.sum(axis=2) > 0).astype(np.float32)[..., None]
    overlay = (
        q_np.astype(np.float32) * (1 - WARP_ALPHA * blend_mask)
        + warped.astype(np.float32) * WARP_ALPHA * blend_mask
    ).clip(0, 255).astype(np.uint8)

    # Draw correspondence lines on a side-by-side canvas; inliers coloured, outliers grey.
    canvas_w = orig_w + q_w
    canvas_h = max(orig_h, q_h)
    corresp  = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    corresp[:orig_h, :orig_w] = ref_np
    corresp[:q_h, orig_w:]    = q_np

    for i, (sp, dp, lbl) in enumerate(zip(src_pts, dst_pts, valid_labels)):
        is_inlier = bool(inlier_mask[i]) if inlier_mask is not None else True
        color = label_color[lbl] if is_inlier else (120, 120, 120)
        pt1 = (int(sp[0]), int(sp[1]))
        pt2 = (int(dp[0]) + orig_w, int(dp[1]))
        cv2.line(corresp, pt1, pt2, color, thickness=2, lineType=cv2.LINE_AA)
        cv2.circle(corresp, pt1, MARKER_RADIUS, color, -1)
        cv2.circle(corresp, pt2, MARKER_RADIUS, color, -1)

    # ── 3-panel figure ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(corresp)
    axes[0].set_title(
        f"Correspondences  (ref → \"{qr['name']}\")\n"
        f"inliers {n_inliers}/{len(src_pts)}  (grey = outlier)",
        fontsize=10,
    )
    axes[0].axis("off")

    axes[1].imshow(warped)
    axes[1].set_title("Reference warped into query frame", fontsize=10)
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title(f"Overlay  (warped α={WARP_ALPHA:.0%})", fontsize=10)
    axes[2].axis("off")

    legend_handles = [
        mpatches.Patch(color=[c / 255 for c in label_color[lbl]], label=lbl)
        for lbl in valid_labels
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=len(valid_labels),
               fontsize=9, framealpha=0.85, bbox_to_anchor=(0.5, -0.04))
    plt.suptitle(
        f"Homography  |  DINOv{DINO_VERSION[1]}-{DINO_SIZE}  |  debias={DEBIAS}",
        fontsize=12, y=1.02,
    )
    plt.tight_layout()
    plt.show()